# Notes to self:
Currently, I am able to get the model to sample with batch dimensions and I am able to use the following internal methods:

* `sample_unconditional_prior`
* `sample_conditional_prior`
* `sample_unconditional_posterior`
* `sample_conditional_posterior`
* `forecast`
* `sample_filter_outputs`
* `sample_statespace_matrices`
* `impulse_response_function`


In [1]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import pytensor.tensor as pt

from pymc_extras.statespace.core.statespace import PyMCStateSpace
from pymc_extras.statespace.filters import StandardFilter, KalmanSmoother

from pymc_extras.statespace.core.properties import (
    Parameter,
    State,
    Shock,
    Coord,
)
from pymc_extras.statespace.utils.constants import ALL_STATE_DIM, ALL_STATE_AUX_DIM, SHOCK_DIM

In [2]:
class AutoRegressiveThree(PyMCStateSpace):
    def __init__(self, mode: str):
        k_states = 3  # size of the state vector x
        k_posdef = 1  # number of shocks (size of the state covariance matrix Q)
        k_endog = 1  # number of observed states
        # batch_size = 2

        super().__init__(
            k_endog=k_endog,
            k_states=k_states,
            k_posdef=k_posdef,
            # batch_size=batch_size,
            mode=mode,
        )

    def make_symbolic_graph(self):
        x0 = self.make_and_register_variable("x0", shape=(3,))
        P0 = self.make_and_register_variable("P0", shape=(3, 3))

        ar_params = self.make_and_register_variable("ar_params", shape=(3,))
        sigma_x = self.make_and_register_variable("sigma_x", shape=(1,))

        self.ssm["transition", :, :] = np.eye(3, k=-1)
        self.ssm["selection", 0, 0] = 1
        self.ssm["design", 0, 0] = 1

        self.ssm["initial_state", :] = x0
        self.ssm["initial_state_cov", :, :] = P0
        self.ssm["transition", 0, :] = ar_params
        self.ssm["state_cov", :, :] = sigma_x

    def set_parameters(self):
        # Only the "name" parameter is required here. "Shape" is only used when printing the
        # model requirements table. "Dims" are used to link variables to coords.
        x0 = Parameter(name="x0", shape=(3,), dims=(ALL_STATE_DIM,))
        P0 = Parameter(name="P0", shape=(3, 3), dims=(ALL_STATE_DIM, ALL_STATE_AUX_DIM))

        ar_params = Parameter(
            name="ar_params", shape=(3,), dims=("ar_lags",), constraints="Stationary, please :)"
        )
        sigma_x = Parameter(name="sigma_x", shape=(1,), dims=(SHOCK_DIM,))
        return x0, P0, ar_params, sigma_x

    def set_states(self):
        # To get a name on the observed, we make an observed state
        ts1 = State(name="ts1", observed=True)

        # Since the three hidden states are lags of the data, i'll call them L1, L2 L3
        L1 = State(name="L1.data", observed=False)
        L2 = State(name="L2.data", observed=False)
        L3 = State(name="L3.data", observed=False)

        return ts1, L1, L2, L3

    def set_shocks(self):
        # There is one shock, called the "innovations" in the literature, so i'll go with that
        innovation = Shock(name="innovations")
        return innovation

    def set_coords(self):
        # This function sets up the coords dictionary used by pm.Model. The parent class has a helper
        # self.default_coords() that makes the coords that are always expected by a statespace model --
        # stuff like state, shock, etc.

        # You need to give one Coord object per dimension used among the Parameter objects you made a

        default_coords = self.default_coords()
        ar_coord = Coord(dimension="ar_lags", labels=(1, 2, 3))
        return *default_coords, ar_coord

In [3]:
ar3 = AutoRegressiveThree(mode="NUMBA")

                          Model Requirements                           
                                                                       
  Variable    Shape    Constraints                         Dimensions  
 ───────────────────────────────────────────────────────────────────── 
  x0          (3,)                                         ('state',)  
  P0          (3, 3)                           ('state', 'state_aux')  
  ar_params   (3,)     Stationary, please :)             ('ar_lags',)  
  sigma_x     (1,)                                         ('shock',)  
                                                                       
 These parameters should be assigned priors inside a PyMC model block  
           before calling the build_statespace_graph method.           

In [4]:
data = np.random.normal(0, 1, size=(100, 2))

In [5]:
batched_data = data.reshape(2, 100, 1)

In [6]:
# Not vectorized
with pm.Model(coords=ar3.coords) as pymc_mod:
    x0 = pm.Deterministic("x0", pt.zeros((3,)), dims=("state"))
    P0 = pm.Deterministic("P0", pt.eye(3) * 10, dims=("state", "state_aux"))
    ar_params = pm.Normal("ar_params", shape=(3,), dims=("state"))

    sigma_x = pm.Exponential("sigma_x", 1, shape=(1,), dims=("shock"))

    ar3.build_statespace_graph(data=data[:, [1]])

/Users/dekermanjian/Desktop/Open_Source_Contributions/pymc-extras/pymc_extras/statespace/utils/data_tools.py:78: UserWarning: No time index found on the supplied data. A simple range index will be automatically generated.
  warnings.warn(NO_TIME_INDEX_WARNING)


In [6]:
# Vectorized
with pm.Model(coords=ar3.coords | {"batch": ["batch_1", "batch_2"]}) as pymc_mod:
    x0 = pm.Deterministic("x0", pt.zeros((2, 3)), dims=("batch", "state"))
    P0 = pm.Deterministic(
        "P0", pt.tile(pt.eye(3) * 1, (2, 1, 1)), dims=("batch", "state", "state_aux")
    )
    ar_params = pm.Normal("ar_params", dims=("batch", "state"))

    sigma_x = pm.Exponential("sigma_x", 1, dims=("batch", "shock"))

    ar3.build_statespace_graph(data=batched_data)

/Users/dekermanjian/Desktop/Open_Source_Contributions/pymc-extras/pymc_extras/statespace/utils/data_tools.py:78: UserWarning: No time index found on the supplied data. A simple range index will be automatically generated.
  warnings.warn(NO_TIME_INDEX_WARNING)


In [ ]:
matrices = ar3.unpack_statespace()

In [ ]:
matrices_2 = ar3._unpack_statespace_with_placeholders()

In [ ]:
[(m.name, m.type.shape) for m in matrices_2]

In [ ]:
[(m.name, m.type.shape) for m in matrices]

In [ ]:
type(matrices[0])

In [ ]:
def infer_batch_dimensions(
    core_matrices: list[pt.TensorVariable],
    subbed_matrices: list[pt.TensorVariable],
) -> tuple[int | None, ...]:
    inferred_batch_dims = ()

    for core_matrix, sub_matrix in zip(core_matrices, subbed_matrices):
        core_shape = core_matrix.type.shape
        sub_shape = sub_matrix.type.shape

        if len(sub_shape) < len(core_shape):
            raise ValueError(
                f"Subbed matrix has fewer dims than core matrix: " f"{sub_shape} vs {core_shape}"
            )

        # Verify trailing/core dimensions match
        trailing_shape = sub_shape[-len(core_shape) :]

        for core_dim, sub_dim in zip(core_shape, trailing_shape):
            if core_dim is not None and sub_dim is not None and core_dim != sub_dim:
                raise ValueError(f"Core dimension mismatch: " f"{core_shape} vs {sub_shape}")

        batch_dims = sub_shape[: -len(core_shape)]

        # Skip matrices with no batch dimensions
        if len(batch_dims) == 0:
            continue

        # First batched tensor establishes the batch shape
        if len(inferred_batch_dims) == 0:
            inferred_batch_dims = batch_dims
            continue

        # Validate consistency
        if len(batch_dims) != len(inferred_batch_dims):
            raise ValueError(f"Inconsistent batch rank: " f"{batch_dims} vs {inferred_batch_dims}")

        merged_dims = []

        for inferred_dim, new_dim in zip(inferred_batch_dims, batch_dims):
            if inferred_dim is None:
                merged_dims.append(new_dim)
            elif new_dim is None:
                merged_dims.append(inferred_dim)
            elif inferred_dim == new_dim:
                merged_dims.append(inferred_dim)
            else:
                raise ValueError(
                    f"Inconsistent batch dimensions: " f"{batch_dims} vs {inferred_batch_dims}"
                )

        inferred_batch_dims = tuple(merged_dims)

    return inferred_batch_dims

In [ ]:
infer_batch_dimensions(core_matrices=matrices_2, subbed_matrices=matrices)

In [7]:
with pymc_mod:
    idata = pm.sample(tune=200, draws=200, compile_kwargs={"mode": "NUMBA"})

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [ar_params, sigma_x]


Output()

Sampling 4 chains for 200 tune and 200 draw iterations (800 + 800 draws total) took 6 seconds.


In [8]:
ar3.impulse_response_function(idata)

Sampling: [initial_shock]


Output()

<xarray.DataTree 'posterior_predictive'>
Group: /posterior_predictive
    Dimensions:  (chain: 4, draw: 200, batch: 2, time: 40, state: 3)
    Coordinates:
      * chain    (chain) int64 32B 0 1 2 3
      * draw     (draw) int64 2kB 0 1 2 3 4 5 6 7 ... 193 194 195 196 197 198 199
      * batch    (batch) <U7 56B 'batch_1' 'batch_2'
      * time     (time) int64 320B 0 1 2 3 4 5 6 7 8 ... 31 32 33 34 35 36 37 38 39
      * state    (state) <U7 84B 'L1.data' 'L2.data' 'L3.data'
    Data variables:
        irf      (chain, draw, batch, time, state) float64 2MB 2.659 ... 4.025e-13
    Attributes:
        created_at:                 2026-05-29T00:26:42.616309+00:00
        creation_library:           ArviZ
        creation_library_version:   1.1.0
        creation_library_language:  Python
        inference_library:          pymc
        inference_library_version:  6.0.0
        sample_dims:                ['chain', 'draw']

In [9]:
ar3.sample_statespace_matrices(idata, matrix_names=["T"])

Sampling: []


Output()

<xarray.DataTree>
Group: /
├── Group: /posterior_predictive
│       Dimensions:    (chain: 4, draw: 200, batch: 2, state: 3, state_aux: 3)
│       Coordinates:
│         * chain      (chain) int64 32B 0 1 2 3
│         * draw       (draw) int64 2kB 0 1 2 3 4 5 6 7 ... 193 194 195 196 197 198 199
│         * batch      (batch) <U7 56B 'batch_1' 'batch_2'
│         * state      (state) <U7 84B 'L1.data' 'L2.data' 'L3.data'
│         * state_aux  (state_aux) <U7 84B 'L1.data' 'L2.data' 'L3.data'
│       Data variables:
│           T          (chain, draw, batch, state, state_aux) float64 115kB 0.06838 ....
│       Attributes:
│           created_at:                 2026-05-29T00:26:44.280981+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.1.0
│           creation_library_language:  Python
│           inference_library:          pymc
│           inference_library_version:  6.0.0
│           sample_dims:                ['chain', 'draw']
└── Group: /observed_data
        Attributes:
            created_at:                 2026-05-29T00:26:44.282187+00:00
            creation_library:           ArviZ
            creation_library_version:   1.1.0
            creation_library_language:  Python
            inference_library:          pymc
            inference_library_version:  6.0.0
            sample_dims:                []

In [10]:
ar3.sample_filter_outputs(idata, filter_output_names=["filtered_covariances"])

/Users/dekermanjian/Desktop/Open_Source_Contributions/pymc-extras/pymc_extras/statespace/utils/data_tools.py:78: UserWarning: No time index found on the supplied data. A simple range index will be automatically generated.
  warnings.warn(NO_TIME_INDEX_WARNING)
Sampling: []


Output()

<xarray.DataTree>
Group: /
├── Group: /posterior_predictive
│       Dimensions:               (chain: 4, draw: 200, batch: 2, time: 100, state: 3,
│                                  state_aux: 3)
│       Coordinates:
│         * chain                 (chain) int64 32B 0 1 2 3
│         * draw                  (draw) int64 2kB 0 1 2 3 4 5 ... 195 196 197 198 199
│         * batch                 (batch) int64 16B 0 1
│         * time                  (time) int64 800B 0 1 2 3 4 5 6 ... 94 95 96 97 98 99
│         * state                 (state) <U7 84B 'L1.data' 'L2.data' 'L3.data'
│         * state_aux             (state_aux) <U7 84B 'L1.data' 'L2.data' 'L3.data'
│       Data variables:
│           filtered_covariances  (chain, draw, batch, time, state, state_aux) float64 12MB ...
│       Attributes:
│           created_at:                 2026-05-29T00:26:48.352615+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.1.0
│           creation_library_language:  Python
│           inference_library:          pymc
│           inference_library_version:  6.0.0
│           sample_dims:                ['chain', 'draw']
├── Group: /observed_data
│       Attributes:
│           created_at:                 2026-05-29T00:26:48.353653+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.1.0
│           creation_library_language:  Python
│           inference_library:          pymc
│           inference_library_version:  6.0.0
│           sample_dims:                []
└── Group: /constant_data
        Dimensions:         (batch: 2, time: 100, observed_state: 1)
        Coordinates:
          * batch           (batch) int64 16B 0 1
          * time            (time) int64 800B 0 1 2 3 4 5 6 7 ... 93 94 95 96 97 98 99
          * observed_state  (observed_state) <U3 12B 'ts1'
        Data variables:
            data            (batch, time, observed_state) float64 2kB -0.9498 ... 0.3309
        Attributes:
            created_at:                 2026-05-29T00:26:48.354952+00:00
            creation_library:           ArviZ
            creation_library_version:   1.1.0
            creation_library_language:  Python
            inference_library:          pymc
            inference_library_version:  6.0.0
            sample_dims:                []

In [11]:
ar3.forecast(idata, periods=10)

No start date provided. Using the last date in the data index. To silence this warning, explicitly pass a start date or set verbose = False
/Users/dekermanjian/Desktop/Open_Source_Contributions/pymc-extras/pymc_extras/statespace/utils/data_tools.py:78: UserWarning: No time index found on the supplied data. A simple range index will be automatically generated.
  warnings.warn(NO_TIME_INDEX_WARNING)
Sampling: [forecast_combined]


Output()

<xarray.DataTree 'posterior_predictive'>
Group: /posterior_predictive
    Dimensions:            (chain: 4, draw: 200, time: 10, batch: 2, state: 3,
                            observed_state: 1)
    Coordinates:
      * chain              (chain) int64 32B 0 1 2 3
      * draw               (draw) int64 2kB 0 1 2 3 4 5 ... 194 195 196 197 198 199
      * time               (time) int64 80B 100 101 102 103 104 105 106 107 108 109
      * batch              (batch) <U7 56B 'batch_1' 'batch_2'
      * state              (state) <U7 84B 'L1.data' 'L2.data' 'L3.data'
      * observed_state     (observed_state) <U3 12B 'ts1'
    Data variables:
        forecast_latent    (chain, draw, time, batch, state) float64 384kB -1.179...
        forecast_observed  (chain, draw, time, batch, observed_state) float64 128kB ...
    Attributes:
        created_at:                 2026-05-29T00:26:56.955319+00:00
        creation_library:           ArviZ
        creation_library_version:   1.1.0
        creation_library_language:  Python
        inference_library:          pymc
        inference_library_version:  6.0.0
        sample_dims:                ['chain', 'draw']

In [12]:
post = ar3.sample_conditional_posterior(idata, mvn_method="cholesky")

/Users/dekermanjian/Desktop/Open_Source_Contributions/pymc-extras/pymc_extras/statespace/utils/data_tools.py:78: UserWarning: No time index found on the supplied data. A simple range index will be automatically generated.
  warnings.warn(NO_TIME_INDEX_WARNING)
Sampling: [filtered_posterior, filtered_posterior_observed, predicted_posterior, predicted_posterior_observed, smoothed_posterior, smoothed_posterior_observed]


Output()

In [13]:
with pymc_mod:
    prior = pm.sample_prior_predictive(compile_kwargs={"mode": "NUMBA"})

Sampling: [ar_params, obs, sigma_x]


In [14]:
ar3.sample_conditional_prior(prior, mvn_method="cholesky")

/Users/dekermanjian/Desktop/Open_Source_Contributions/pymc-extras/pymc_extras/statespace/utils/data_tools.py:78: UserWarning: No time index found on the supplied data. A simple range index will be automatically generated.
  warnings.warn(NO_TIME_INDEX_WARNING)
Sampling: [filtered_prior, filtered_prior_observed, predicted_prior, predicted_prior_observed, smoothed_prior, smoothed_prior_observed]


Output()

<xarray.DataTree 'posterior_predictive'>
Group: /posterior_predictive
    Dimensions:                   (chain: 1, draw: 500, batch: 2, time: 100,
                                   state: 3, observed_state: 1)
    Coordinates:
      * chain                     (chain) int64 8B 0
      * draw                      (draw) int64 4kB 0 1 2 3 4 ... 495 496 497 498 499
      * batch                     (batch) <U7 56B 'batch_1' 'batch_2'
      * time                      (time) int64 800B 0 1 2 3 4 5 ... 95 96 97 98 99
      * state                     (state) <U7 84B 'L1.data' 'L2.data' 'L3.data'
      * observed_state            (observed_state) <U3 12B 'ts1'
    Data variables:
        filtered_prior            (chain, draw, batch, time, state) float64 2MB -...
        filtered_prior_observed   (chain, draw, batch, time, observed_state) float64 800kB ...
        predicted_prior           (chain, draw, batch, time, state) float64 2MB -...
        predicted_prior_observed  (chain, draw, batch, time, observed_state) float64 800kB ...
        smoothed_prior            (chain, draw, batch, time, state) float64 2MB -...
        smoothed_prior_observed   (chain, draw, batch, time, observed_state) float64 800kB ...
    Attributes:
        created_at:                 2026-05-29T00:27:23.480095+00:00
        creation_library:           ArviZ
        creation_library_version:   1.1.0
        creation_library_language:  Python
        inference_library:          pymc
        inference_library_version:  6.0.0
        sample_dims:                ['chain', 'draw']

In [15]:
ar3.sample_unconditional_prior(prior, mvn_method="cholesky")

Sampling: [prior_combined]


Output()

<xarray.DataTree 'posterior_predictive'>
Group: /posterior_predictive
    Dimensions:         (chain: 1, draw: 500, time: 100, batch: 2, state: 3,
                         observed_state: 1)
    Coordinates:
      * chain           (chain) int64 8B 0
      * draw            (draw) int64 4kB 0 1 2 3 4 5 6 ... 494 495 496 497 498 499
      * time            (time) int64 800B 0 1 2 3 4 5 6 7 ... 93 94 95 96 97 98 99
      * batch           (batch) <U7 56B 'batch_1' 'batch_2'
      * state           (state) <U7 84B 'L1.data' 'L2.data' 'L3.data'
      * observed_state  (observed_state) <U3 12B 'ts1'
    Data variables:
        prior_latent    (chain, draw, time, batch, state) float64 2MB -0.3617 ......
        prior_observed  (chain, draw, time, batch, observed_state) float64 800kB ...
    Attributes:
        created_at:                 2026-05-29T00:27:25.728592+00:00
        creation_library:           ArviZ
        creation_library_version:   1.1.0
        creation_library_language:  Python
        inference_library:          pymc
        inference_library_version:  6.0.0
        sample_dims:                ['chain', 'draw']

In [16]:
unpost = ar3.sample_unconditional_posterior(idata, mvn_method="cholesky")

Sampling: [posterior_combined]


Output()

In [17]:
unpost

<xarray.DataTree 'posterior_predictive'>
Group: /posterior_predictive
    Dimensions:             (chain: 4, draw: 200, time: 100, batch: 2, state: 3,
                             observed_state: 1)
    Coordinates:
      * chain               (chain) int64 32B 0 1 2 3
      * draw                (draw) int64 2kB 0 1 2 3 4 5 ... 194 195 196 197 198 199
      * time                (time) int64 800B 0 1 2 3 4 5 6 ... 93 94 95 96 97 98 99
      * batch               (batch) <U7 56B 'batch_1' 'batch_2'
      * state               (state) <U7 84B 'L1.data' 'L2.data' 'L3.data'
      * observed_state      (observed_state) <U3 12B 'ts1'
    Data variables:
        posterior_latent    (chain, draw, time, batch, state) float64 4MB -1.517 ...
        posterior_observed  (chain, draw, time, batch, observed_state) float64 1MB ...
    Attributes:
        created_at:                 2026-05-29T00:27:27.765195+00:00
        creation_library:           ArviZ
        creation_library_version:   1.1.0
        creation_library_language:  Python
        inference_library:          pymc
        inference_library_version:  6.0.0
        sample_dims:                ['chain', 'draw']

# MVN Method

In [ ]:
class AutoRegressive3TwoSeries(PyMCStateSpace):
    def __init__(self, mode: str):
        k_states = 6  # 2 series × 3 lags
        k_posdef = 2  # one innovation per series
        k_endog = 2  # two observed series

        super().__init__(k_endog=k_endog, k_states=k_states, k_posdef=k_posdef, mode=mode)

    def make_symbolic_graph(self):
        x0 = self.make_and_register_variable("x0", shape=(6,))
        P0 = self.make_and_register_variable("P0", shape=(6, 6))

        ar_params = self.make_and_register_variable("ar_params", shape=(2, 3))
        sigma_x = self.make_and_register_variable("sigma_x", shape=(2,))

        T = np.eye(6, k=-1)

        self.ssm["transition", :, :] = T
        self.ssm["transition", 0, 0:3] = ar_params[0]
        self.ssm["transition", 3, 3:6] = ar_params[1]

        self.ssm["selection", 0, 0] = 1
        self.ssm["selection", 3, 1] = 1

        self.ssm["state_cov", :, :] = pt.diag(sigma_x)

        Z = np.zeros((2, 6))
        Z[0, 0] = 1
        Z[1, 3] = 1
        self.ssm["design", :, :] = Z

        self.ssm["initial_state", :] = x0
        self.ssm["initial_state_cov", :, :] = P0

    def set_parameters(self):
        x0 = Parameter(name="x0", shape=(6,), dims=(ALL_STATE_DIM,))
        P0 = Parameter(name="P0", shape=(6, 6), dims=(ALL_STATE_DIM, ALL_STATE_AUX_DIM))

        ar_params = Parameter(
            name="ar_params",
            shape=(2, 3),
            dims=("observed_state", "ar_lags"),
        )

        sigma_x = Parameter(
            name="sigma_x",
            shape=(2,),
            dims=("observed_state",),
        )

        return x0, P0, ar_params, sigma_x

    def set_states(self):
        # Observed states
        ts1 = State(name="ts1", observed=True)
        ts2 = State(name="ts2", observed=True)

        # Series 1 states
        L1_s1 = State(name="L1.ts1", observed=False)
        L2_s1 = State(name="L2.ts1", observed=False)
        L3_s1 = State(name="L3.ts1", observed=False)

        # Series 2 states
        L1_s2 = State(name="L1.ts2", observed=False)
        L2_s2 = State(name="L2.ts2", observed=False)
        L3_s2 = State(name="L3.ts2", observed=False)

        return (
            ts1,
            ts2,
            L1_s1,
            L2_s1,
            L3_s1,
            L1_s2,
            L2_s2,
            L3_s2,
        )

    def set_shocks(self):
        eps1 = Shock(name="innovation.ts1")
        eps2 = Shock(name="innovation.ts2")
        return eps1, eps2

    def set_coords(self):
        default_coords = self.default_coords()
        ar_coord = Coord(dimension="ar_lags", labels=(1, 2, 3))
        return *default_coords, ar_coord

In [ ]:
ar3.coords

In [ ]:
ar3.ssm["design"].eval()

In [ ]:
ar3 = AutoRegressive3TwoSeries(mode="NUMBA")

In [ ]:
with pm.Model(coords=ar3.coords) as pymc_mod:
    x0 = pm.Deterministic("x0", pt.zeros(6), dims=["state"])
    P0 = pm.Deterministic("P0", pt.eye(6) * 10, dims=["state", "state_aux"])

    # global mean per lag
    rho_global = pm.Normal("rho_global", 0.0, 0.5, dims=["ar_lags"])
    tau = pm.Exponential("tau", 2.0, dims=["ar_lags"])

    ar_offset = pm.Normal("ar_offset", 0.0, 1.0, dims=["observed_state", "ar_lags"])

    ar_params = pm.Deterministic(
        "ar_params",
        rho_global + tau * ar_offset,
        dims=["observed_state", "ar_lags"],
    )

    sigma_x = pm.Exponential("sigma_x", 1.0, dims=["observed_state"])

    ar3.build_statespace_graph(data=data)
    idata = pm.sample(compile_kwargs={"mode": "NUMBA"})

In [ ]:
pymc_mod.to_graphviz()

# MISC